# Préparation des données - Nettoyage et standardisation

---

## Objectif du notebook

Ce notebook a pour objectif de :

- Charger le dataset IBM HR Analytics Employee Attrition & Performance,
- Effectuer les contrôles de qualité de base (types, valeurs manquantes, doublons),
- Standardiser les formats des variables,
- Produire un **dataset propre et exploitable** pour les analyses exploratoires ultérieures.

Le jeu de données final est enregistré au format **Parquet** dans : `data/processed/employees_clean.parquet`



## 1. Import des librairies

---

In [1]:
import kagglehub
import pandas as pd
import pyarrow
import numpy as np
import os
from IPython.display import display_html

from pathlib import Path

Ce projet utilise des chemins relatifs afin de garantir la reproductibilité des notebooks.

In [2]:
# Définition des chemins

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

## 2. Chargement des données

---

In [3]:
# 1. Téléchargement du dataset
path = kagglehub.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")
# print("Données téléchargées ici :", path)

# 2. Lister les fichiers
print("\nFichiers disponibles :")
for fichier in os.listdir(path):
    print(" -", fichier)


# 3. Charger le CSV principal
csv_path = os.path.join(path, "WA_Fn-UseC_-HR-Employee-Attrition.csv")
df = pd.read_csv(csv_path)

# 4. Afficher un aperçu
print("\nAperçu du dataset :")
display(df.head(2))




Fichiers disponibles :
 - WA_Fn-UseC_-HR-Employee-Attrition.csv

Aperçu du dataset :


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7


## 3. Contrôles qualité initiaux

---

### 3.1 Structure générale du dataset

Ces éléments permettent de vérifier la diversité des variables et d'identifier d'éventuelles colonnes constantes ou peu informatives.


In [4]:
display(f"Le dataset contient {df.shape[0]} lignes et {df.shape[1]} colonnes")

'Le dataset contient 1470 lignes et 35 colonnes'

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [6]:
df.dtypes

Age                          int64
Attrition                   object
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EmployeeCount                int64
EmployeeNumber               int64
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
Over18                      object
OverTime                    object
PercentSalaryHike            int64
PerformanceRating            int64
RelationshipSatisfaction     int64
StandardHours                int64
StockOptionLevel             int64
TotalWorkingYears   

In [7]:
# Colonnes numériques
df.select_dtypes(include="number").describe()

,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
count,1470.000000,1470.000000,1470.000000,1470.000000,1470.0,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,...,1470.000000,1470.0,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000,1470.000000
mean,36.923810,802.485714,9.192517,2.912925,1.0,1024.865306,2.721769,65.891156,2.729932,2.063946,...,2.712245,80.0,0.793878,11.279592,2.799320,2.761224,7.008163,4.229252,2.187755,4.123129
std,9.135373,403.509100,8.106864,1.024165,0.0,602.024335,1.093082,20.329428,0.711561,1.106940,...,1.081209,0.0,0.852077,7.780782,1.289271,0.706476,6.126525,3.623137,3.222430,3.568136
min,18.000000,102.000000,1.000000,1.000000,1.0,1.000000,1.000000,30.000000,1.000000,1.000000,...,1.000000,80.0,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,30.000000,465.000000,2.000000,2.000000,1.0,491.250000,2.000000,48.000000,2.000000,1.000000,...,2.000000,80.0,0.000000,6.000000,2.000000,2.000000,3.000000,2.000000,0.000000,2.000000
50%,36.000000,802.000000,7.000000,3.000000,1.0,1020.500000,3.000000,66.000000,3.000000,2.000000,...,3.000000,80.0,1.000000,10.000000,3.000000,3.000000,5.000000,3.000000,1.000000,3.000000
75%,43.000000,1157.000000,14.000000,4.000000,1.0,1555.750000,4.000000,83.750000,3.000000,3.000000,...,4.000000,80.0,1.000000,15.000000,3.000000,3.000000,9.000000,7.000000,3.000000,7.000000
max,60.000000,1499.000000,29.000000,5.000000,1.0,2068.000000,4.000000,100.000000,4.000000,5.000000,...,4.000000,80.0,3.000000,40.000000,6.000000,4.000000,40.000000,18.000000,15.000000,17.000000


In [8]:
# Colonnes catégorielles et numériques uniques

num = df.select_dtypes(include="number").nunique().sort_values(ascending=False)
cat = df.select_dtypes(exclude="number").nunique().sort_values(ascending=False)

num_tbl = num.rename("Uniques").to_frame()
num_tbl.index.name = "Numérique"

cat_tbl = cat.rename("Uniques").to_frame()
cat_tbl.index.name = "Catégorielle"

html = f"""
<div style="display:flex; gap:40px; align-items:flex-start;">
  <div>
    <h4>Colonnes numériques</h4>
    {num_tbl.to_html()}
  </div>
  <div>
    <h4>Colonnes catégorielles</h4>
    {cat_tbl.to_html()}
  </div>
</div>
"""
display_html(html, raw=True)


,Uniques
Numérique,
EmployeeNumber,1470
MonthlyRate,1427
MonthlyIncome,1349
DailyRate,886
HourlyRate,71
Age,43
TotalWorkingYears,40
YearsAtCompany,37
DistanceFromHome,29


### 3.2 Valeurs manquantes


In [9]:
# Détecter les valeurs manquantes
display(df.isnull().mean())

# Vérifier si toutes les colonnes ne contiennent aucune valeur manquante
if (df.isnull().mean() == 0).all():
    print("Il n'y a aucune valeur manquante")
else:
    print("Il existe des valeurs manquantes dans le dataframe")



Age                         0.0
Attrition                   0.0
BusinessTravel              0.0
DailyRate                   0.0
Department                  0.0
DistanceFromHome            0.0
Education                   0.0
EducationField              0.0
EmployeeCount               0.0
EmployeeNumber              0.0
EnvironmentSatisfaction     0.0
Gender                      0.0
HourlyRate                  0.0
JobInvolvement              0.0
JobLevel                    0.0
JobRole                     0.0
JobSatisfaction             0.0
MaritalStatus               0.0
MonthlyIncome               0.0
MonthlyRate                 0.0
NumCompaniesWorked          0.0
Over18                      0.0
OverTime                    0.0
PercentSalaryHike           0.0
PerformanceRating           0.0
RelationshipSatisfaction    0.0
StandardHours               0.0
StockOptionLevel            0.0
TotalWorkingYears           0.0
TrainingTimesLastYear       0.0
WorkLifeBalance             0.0
YearsAtC

Il n'y a aucune valeur manquante


Aucune valeur manquante n'est observée dans le dataset.

### 3.3 Doublons


In [10]:

display(f'La somme des doublons est de {df.duplicated().sum()}')

# Aucun doublon

'La somme des doublons est de 0'

## 4. Nettoyage et standardisation

---

### 4.1 Colonnes catégorielles

Les variables catégorielles sont converties en type `string` et leurs valeurs sont harmonisées (suppression des espaces superflus et normalisation de la casse).


In [11]:
# Colonnes catégorielles
colonne_str = df.select_dtypes(exclude="number").columns

# Convertir en string 
df[colonne_str] = df[colonne_str].astype("string")
df.select_dtypes(exclude="number").info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Attrition       1470 non-null   string
 1   BusinessTravel  1470 non-null   string
 2   Department      1470 non-null   string
 3   EducationField  1470 non-null   string
 4   Gender          1470 non-null   string
 5   JobRole         1470 non-null   string
 6   MaritalStatus   1470 non-null   string
 7   Over18          1470 non-null   string
 8   OverTime        1470 non-null   string
dtypes: string(9)
memory usage: 103.5 KB


In [12]:
# Uniformiser 
df[colonne_str] = df[colonne_str].apply(lambda x:x.str.strip())

# Vérification des catégories
for col in colonne_str:
    print(f'\n{col}\n{df[col].unique()}')
   


Attrition
<StringArray>
['Yes', 'No']
Length: 2, dtype: string

BusinessTravel
<StringArray>
['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
Length: 3, dtype: string

Department
<StringArray>
['Sales', 'Research & Development', 'Human Resources']
Length: 3, dtype: string

EducationField
<StringArray>
[   'Life Sciences',            'Other',          'Medical',
        'Marketing', 'Technical Degree',  'Human Resources']
Length: 6, dtype: string

Gender
<StringArray>
['Female', 'Male']
Length: 2, dtype: string

JobRole
<StringArray>
[          'Sales Executive',        'Research Scientist',
     'Laboratory Technician',    'Manufacturing Director',
 'Healthcare Representative',                   'Manager',
      'Sales Representative',         'Research Director',
           'Human Resources']
Length: 9, dtype: string

MaritalStatus
<StringArray>
['Single', 'Married', 'Divorced']
Length: 3, dtype: string

Over18
<StringArray>
['Y']
Length: 1, dtype: string

OverTime
<StringArray>


### 4.2 Vérification des colonnes numériques

Analyse des valeurs minimales / maximales, des distributions et détection de colonnes constantes.


In [13]:
# Colonnes numériques
colonne_num = df.select_dtypes(include="number").columns

# Vérification de colonnes numériques spécifiques
display(f"MonthlyIncome min = {df['MonthlyIncome'].min()} max = {df['MonthlyIncome'].max()}")
display(f"PercentSalaryHike min = {df['PercentSalaryHike'].min()} max = {df['PercentSalaryHike'].max()}")

# Détection des colonnes numériques constantes
for col in colonne_num:
    if df[col].nunique() == 1:
         print(f"- {col} est constante ({df[col].unique()[0]})")


'MonthlyIncome min = 1009 max = 19999'

'PercentSalaryHike min = 11 max = 25'

- EmployeeCount est constante (1)
- StandardHours est constante (80)


Les colonnes constantes identifiées pourront être exclues des analyses ultérieures.

## 5. Conclusion et sauvegarde

---

## Conclusion

À l'issue de cette étape :

- le dataset ne présente ni valeurs manquantes ni doublons,
- les types de données ont été harmonisés,
- les variables catégorielles ont été standardisées,
- aucune anomalie bloquante n'a été détectée.

Le dataset nettoyé est enregistré sous le nom : `employees_clean.parquet`

Il constitue la base de travail pour l'ensemble des analyses par axe.


In [14]:
# Sauvegarde en Parquet du dataset nettoyé
df_clean = df.copy()
df_clean.to_parquet(PROCESSED_DATA_DIR/"employees_clean.parquet", index=False)
